# LAB 10 — Tokenization, Offset Mapping & Answer Span Alignment (Mục 10–16)


In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Kiểm chứng Calculation Exercise 3 (Mục 12)
question = "What is the capital of France?"
context = "The capital of France is Paris."
answer_text = "Paris"
answer_start = context.index(answer_text)
answer_end = answer_start + len(answer_text)

encoded = tokenizer(
    question,
    context,
    return_offsets_mapping=True,
    return_attention_mask=True,
)

print(f"Character span of '{answer_text}': ({answer_start}, {answer_end})")
print("Keys returned:", list(encoded.keys()))

tokens = tokenizer.convert_ids_to_tokens(encoded["input_ids"])
offsets = encoded["offset_mapping"]
seq_ids = encoded.sequence_ids()

for idx, (tok, off, sid) in enumerate(zip(tokens, offsets, seq_ids)):
    print(f"idx={idx:2d} | seq_id={str(sid):4s} | token={tok:10s} | offset={off}")

## Hàm `prepare_train_features` với Sliding Window & Xử lý 4 trường hợp biên (Mục 15 & 16)
1. Answer bắt đầu giữa subword token.
2. Answer kết thúc giữa subword token.
3. Answer bị truncation ở ranh giới window.
4. Answer nằm hoàn toàn ngoài window hiện tại.

In [ ]:
def prepare_train_features(examples, max_length=384, stride=128):
    questions = [q.strip() for q in examples["question"]]

    tokenized_examples = tokenizer(
        questions,
        examples["context"],
        truncation="only_second",
        max_length=max_length,
        stride=stride,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    sample_mapping = tokenized_examples.pop("overflow_to_sample_mapping")
    offset_mapping = tokenized_examples.pop("offset_mapping")

    tokenized_examples["start_positions"] = []
    tokenized_examples["end_positions"] = []

    for i, offsets in enumerate(offset_mapping):
        input_ids = tokenized_examples["input_ids"][i]
        cls_index = input_ids.index(tokenizer.cls_token_id)
        sequence_ids = tokenized_examples.sequence_ids(i)

        sample_index = sample_mapping[i]
        answers = examples["answers"][sample_index]

        if len(answers["answer_start"]) == 0:
            tokenized_examples["start_positions"].append(cls_index)
            tokenized_examples["end_positions"].append(cls_index)
            continue

        answer_start = answers["answer_start"][0]
        answer_end = answer_start + len(answers["text"][0])

        token_start_index = 0
        while sequence_ids[token_start_index] != 1:
            token_start_index += 1

        token_end_index = len(input_ids) - 1
        while sequence_ids[token_end_index] != 1:
            token_end_index -= 1

        # Kiểm tra answer bị truncation hoặc nằm ngoài window hiện tại
        if not (offsets[token_start_index][0] <= answer_start and offsets[token_end_index][1] >= answer_end):
            tokenized_examples["start_positions"].append(cls_index)
            tokenized_examples["end_positions"].append(cls_index)
        else:
            # Xử lý answer bắt đầu ở đầu hoặc giữa subword token
            while token_start_index < len(offsets) and offsets[token_start_index][0] <= answer_start:
                token_start_index += 1
            start_token = token_start_index - 1

            # Xử lý answer kết thúc ở cuối hoặc giữa subword token
            while offsets[token_end_index][1] >= answer_end:
                token_end_index -= 1
            end_token = token_end_index + 1

            tokenized_examples["start_positions"].append(start_token)
            tokenized_examples["end_positions"].append(end_token)

    return tokenized_examples

In [ ]:
# Kiểm thử 4 trường hợp biên (Mục 16)
long_prefix = "Background information about European geography and history. " * 25
test_context = long_prefix + "The capital of France is uncharacteristically called Paris today."

# Case 1 & 2: Subword boundary ('characteristically' bên trong 'uncharacteristically')
sub_ans = "characteristically"
sub_start = test_context.index(sub_ans)

# Case 3 & 4: Answer nằm xa cuối context dài -> tạo nhiều windows với max_length=64, stride=16
far_ans = "Paris"
far_start = test_context.index(far_ans)

batch_examples = {
    "question": [
        "How is it called?",
        "What is the capital of France?"
    ],
    "context": [test_context, test_context],
    "answers": [
        {"text": [sub_ans], "answer_start": [sub_start]},
        {"text": [far_ans], "answer_start": [far_start]},
    ]
}

features = prepare_train_features(batch_examples, max_length=64, stride=16)
print("Total sliding windows generated:", len(features["input_ids"]))
for w_idx in range(len(features["input_ids"])):
    s_pos = features["start_positions"][w_idx]
    e_pos = features["end_positions"][w_idx]
    toks = tokenizer.convert_ids_to_tokens(features["input_ids"][w_idx][s_pos:e_pos+1])
    print(f"Window {w_idx:2d}: start={s_pos:2d}, end={e_pos:2d} -> tokens={toks}")